In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from sklearn.model_selection import train_test_split
import timm

In [2]:
DATASET_PATH = r"D:\ML-PROJECTS\long-hair detection\datasets\age\UTKFace"

In [3]:
data = []

for file in os.listdir(DATASET_PATH):

    if file.endswith(".jpg"):

        try:

            age = int(file.split("_")[0])

            data.append([
                os.path.join(DATASET_PATH,file),
                age
            ])

        except:
            pass

df = pd.DataFrame(
    data,
    columns=["image_path","age"]
)

print(df.head())
print("Total:", len(df))

                                          image_path  age
0  D:\ML-PROJECTS\long-hair detection\datasets\ag...  100
1  D:\ML-PROJECTS\long-hair detection\datasets\ag...  100
2  D:\ML-PROJECTS\long-hair detection\datasets\ag...  100
3  D:\ML-PROJECTS\long-hair detection\datasets\ag...  100
4  D:\ML-PROJECTS\long-hair detection\datasets\ag...  100
Total: 23708


In [4]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42
)

In [5]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [6]:
class AgeDataset(Dataset):

    def __init__(self, df, transform=None):

        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        img_path = self.df.loc[idx,"image_path"]

        age = self.df.loc[idx,"age"]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(
            age,
            dtype=torch.float32
        )

In [ ]:
train_loader = DataLoader(
    AgeDataset(
        train_df,
        train_transform
    ),
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    AgeDataset(
        val_df,
        test_transform
    ),
    batch_size=32
)

test_loader = DataLoader(
    AgeDataset(
        test_df,
        test_transform
    ),
    batch_size=s2
)

In [8]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [9]:
model = timm.create_model(
    "efficientnet_b0",
    pretrained=True,
    num_classes=1
)

model = model.to(device)

In [10]:
criterion = nn.SmoothL1Loss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

In [ ]:
EPOCHS = 50

best_mae = float('inf')

patience = 7
counter = 0

for epoch in range(EPOCHS):

    model.train()

    train_loss = 0

    for images, ages in tqdm(train_loader):

        images = images.to(device)
        ages = ages.to(device)

        optimizer.zero_grad()

        outputs = model(images).squeeze()

        loss = criterion(outputs, ages)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    model.eval()

    preds = []
    actuals = []

    with torch.no_grad():

        for images, ages in val_loader:

            images = images.to(device)

            outputs = model(images).squeeze()

            preds.extend(outputs.cpu().numpy())
            actuals.extend(ages.numpy())

    mae = np.mean(
        np.abs(
            np.array(preds) -
            np.array(actuals)
        )
    )

    print(
        f"Epoch {epoch+1}/{EPOCHS} "
        f"MAE={mae:.4f}"
    )

    if mae < best_mae:

        best_mae = mae
        counter = 0

        torch.save(
            model.state_dict(),
            "age_model.pth"
        )

        print(f"Saved Best Model | Best MAE: {best_mae:.4f}")

    else:

        counter += 1

        print(f"No improvement for {counter}/{patience} epochs")

        if counter >= patience:

            print(f"\nEarly Stopping Triggered at Epoch {epoch+1}")
            print(f"Best MAE: {best_mae:.4f}")
            break

100%|██████████| 593/593 [02:15<00:00,  4.37it/s]


Epoch 1/50 MAE=7.1501
Saved Best Model | Best MAE: 7.1501


100%|██████████| 593/593 [02:15<00:00,  4.39it/s]


Epoch 2/50 MAE=6.3845
Saved Best Model | Best MAE: 6.3845


100%|██████████| 593/593 [02:16<00:00,  4.35it/s]


...

Epoch 48/50 MAE=4.7816
Saved Best Model | Best MAE: 4.7816


100%|██████████| 593/593 [02:15<00:00,  4.37it/s]


Epoch 49/50 MAE=4.8349
No improvement for 1/7 epochs


100%|██████████| 593/593 [02:14<00:00,  4.39it/s]


Epoch 50/50 MAE=4.7047
Saved Best Model | Best MAE: 4.7047
